# 🚀 TrafficVision-Transformer — Train STE-TTE (Người 2)

**Notebook này dành cho Người 2.** Chạy từng ô theo thứ tự, đọc hướng dẫn trước mỗi ô.

> ⚠️ Trước khi chạy: **Runtime → Change runtime type → T4 GPU**

## BƯỚC 1 — Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU!')
!nvidia-smi

## BƯỚC 2 — Mount Google Drive

Sẽ hiện popup yêu cầu đăng nhập Google. Nhấn **Connect to Google Drive**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted OK!')

## BƯỚC 3 — Clone code từ GitHub

**Sửa `YOUR_GITHUB_REPO_URL` thành link repo của nhóm.**

Ví dụ: `https://github.com/ten-nhom/TrafficVision-Transformer.git`

Nếu repo **private**, dùng token:
`https://<TOKEN>@github.com/ten-nhom/TrafficVision-Transformer.git`

In [ ]:
import os

GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/TrafficVision-Transformer.git'  # ← SỬA Ở ĐÂY
REPO_DIR = '/content/TrafficVision-Transformer'

if os.path.exists(REPO_DIR):
    print('Repo đã tồn tại, pulling bản mới nhất...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls

## BƯỚC 4 — Cài dependencies

Colab đã có sẵn torch/torchvision, chỉ cần cài thêm `timm`.

In [ ]:
!pip install timm==1.0.3 tqdm scikit-learn -q
print('Dependencies installed OK!')

## BƯỚC 5 — Trỏ đến dataset UIT-ADrone

Dataset phải được upload lên Google Drive theo cấu trúc:
```
My Drive/
└── UIT-ADrone/
    ├── train/
    │   └── frames/   ← chứa các folder scene với .jpg
    └── test/
        ├── frames/   ← chứa các folder scene với .jpg
        └── test_frame_mask/  ← chứa các file .npy label
```

**Chạy ô này để kiểm tra dataset có đúng vị trí không.**

In [ ]:
import os

# ← SỬA đường dẫn nếu bạn để dataset ở thư mục khác trong Drive
DATA_DIR = '/content/drive/MyDrive/UIT-ADrone'

# Kiểm tra cấu trúc
train_path = os.path.join(DATA_DIR, 'train/frames')
test_path  = os.path.join(DATA_DIR, 'test/frames')
label_path = os.path.join(DATA_DIR, 'test/test_frame_mask')

print('Train frames:', os.path.exists(train_path), '→', train_path)
print('Test frames: ', os.path.exists(test_path),  '→', test_path)
print('Labels:      ', os.path.exists(label_path), '→', label_path)

if os.path.exists(train_path):
    scenes = os.listdir(train_path)
    print(f'\nTìm thấy {len(scenes)} scene trong train: {scenes[:5]}...')
else:
    print('\n❌ KHÔNG TÌM THẤY DATASET! Kiểm tra lại đường dẫn DATA_DIR.')

## BƯỚC 6 — Tạo thư mục lưu checkpoint trên Drive

Checkpoint sẽ được lưu thẳng vào Drive để không mất khi Colab disconnect.

In [ ]:
import os

# Thư mục lưu checkpoint trên Drive
DRIVE_CKPT_DIR = '/content/drive/MyDrive/experiments_andt_ADrone_STE_TTE'
os.makedirs(os.path.join(DRIVE_CKPT_DIR, 'checkpoints'), exist_ok=True)

# Symlink để script tự động save vào Drive
LOCAL_SAVE_PATH = '/content/TrafficVision-Transformer/experiments_andt_ADrone_STE_TTE'
if not os.path.exists(LOCAL_SAVE_PATH):
    os.symlink(DRIVE_CKPT_DIR, LOCAL_SAVE_PATH)
    print(f'Symlink created: {LOCAL_SAVE_PATH} → {DRIVE_CKPT_DIR}')
else:
    print(f'Thư mục đã tồn tại: {LOCAL_SAVE_PATH}')

print('Checkpoint sẽ tự động lưu vào Drive!')

## BƯỚC 7 — TRAINING 🚂

Chạy ô này để bắt đầu train. Log sẽ hiện trực tiếp bên dưới.

**Ước tính thời gian:** ~30-90 phút tùy GPU Colab cấp.

> ⚠️ **Đừng đóng tab** trong lúc chạy! Nếu bị disconnect thì checkpoint vẫn còn trên Drive.

In [ ]:
import os
os.chdir('/content/TrafficVision-Transformer')

# Chỉnh DATA_DIR cho khớp với BƯỚC 5
DATA_DIR = '/content/drive/MyDrive/UIT-ADrone'

!python scripts/train_ste_tte.py \
    --train 1 \
    --epochs 5 \
    --batch-size 8 \
    --lr 1e-4 \
    --model-arch b16 \
    --image-size 384 \
    --data-dir {DATA_DIR}

## BƯỚC 8 — Kiểm tra kết quả training

In [ ]:
import os

# Kiểm tra checkpoint đã lưu chưa
ckpt_dir = '/content/drive/MyDrive/experiments_andt_ADrone_STE_TTE/checkpoints'
print('Files trong checkpoints:')
for f in os.listdir(ckpt_dir):
    size_mb = os.path.getsize(os.path.join(ckpt_dir, f)) / 1e6
    print(f'  {f} — {size_mb:.1f} MB')

# Đọc training log
log_path = '/content/TrafficVision-Transformer/training_log_ste_tte.txt'
if os.path.exists(log_path):
    print('\n=== Training Log ===')
    with open(log_path) as f:
        for line in f:
            print(line.strip())
else:
    print('Chưa có log — training chưa xong?')

## BƯỚC 9 — Copy log lên Drive (để không mất sau khi session hết)

In [ ]:
import shutil

src = '/content/TrafficVision-Transformer/training_log_ste_tte.txt'
dst = '/content/drive/MyDrive/experiments_andt_ADrone_STE_TTE/training_log_ste_tte.txt'

if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'Log đã copy lên Drive: {dst}')
else:
    print('Không tìm thấy log file.')

---
## ✅ Xong Ngày 2!

Sau khi train xong, báo cho nhóm:
- Checkpoint đã lưu trong Google Drive: `experiments_andt_ADrone_STE_TTE/checkpoints/best.pth`
- Training log: `experiments_andt_ADrone_STE_TTE/training_log_ste_tte.txt`

**Ngày 3:** Chạy inference + tính AUC (tao sẽ hướng dẫn sau khi Ngày 2 xong).